In [0]:
# Databricks notebook source
# Run after Silver has been successfully built. Rebuilds ONLY Gold tables.
import pyspark.sql.functions as F
from pyspark.sql.types import DecimalType

SCHEMA = 'workspace.retail_pipeline_test'
SILVER = f'{SCHEMA}.silver_sales_v2'
source = spark.table(SILVER)
required = {'InvoiceNo','StockCode','Description','Quantity','InvoiceDate','UnitPrice','CustomerID','Country'}
assert required.issubset(set(source.columns)), f'Missing columns: {required-set(source.columns)}'

# Preserve Silver as-is; remove accounting adjustment invoices only in Gold.
retail = (source.filter(F.col('InvoiceNo').isNotNull())
          .filter(~F.upper(F.trim(F.col('InvoiceNo'))).startswith('A'))
retail = retail.withColumn('Revenue', F.col('Quantity') * F.col('UnitPrice'))
retail = retail.withColumn('IsCancellation', F.upper(F.trim(F.col('InvoiceNo'))).startswith('C'))

# Business contract: C-prefix invoices are cancellations/returns, not purchase orders.
# Verify the dataset's actual business meaning before relying on these KPIs.
sales = retail.filter(~F.col('IsCancellation'))
returns = retail.filter(F.col('IsCancellation'))

# Safeguards before writing tables.
assert retail.filter(F.col('InvoiceDate').isNull()).count() == 0, 'Null dates would lose revenue from monthly report'
assert retail.filter(F.col('Country').isNull()).count() == 0, 'Null countries require an explicit Unknown policy'
assert retail.filter(F.col('Revenue').isNull()).count() == 0, 'Null revenue in Gold input'
assert retail.filter((~F.col('IsCancellation')) & (F.col('Revenue') < 0)).count() == 0, 'Negative sales revenue: investigate'
assert returns.filter(F.col('Revenue') > 0).count() == 0, 'Positive cancellation revenue: investigate'

metrics = retail.agg(F.sum('Revenue').alias('NetRevenue'),
                     F.countDistinct(F.when(F.col('CustomerID').isNotNull() & (F.col('CustomerID') != '-1'), F.col('CustomerID'))).alias('TotalCustomers'))
gross = sales.agg(F.sum('Revenue').alias('GrossRevenue'),
                  F.countDistinct(F.when(F.col('Quantity') > 0, F.col('InvoiceNo'))).alias('TotalOrders'))
return_value = returns.agg((-F.sum('Revenue')).alias('ReturnsValue'))

gold_kpis = (gross.crossJoin(metrics).crossJoin(return_value)
    .withColumn('AOV', F.when(F.col('TotalOrders') > 0,
               F.round(F.col('NetRevenue') / F.col('TotalOrders'), 2))
               .otherwise(F.lit(None).cast(DecimalType(32,2))))
    .select('GrossRevenue','ReturnsValue','NetRevenue','TotalCustomers','TotalOrders','AOV'))

monthly = (retail.withColumn('Month', F.date_format('InvoiceDate','yyyy-MM'))
           .groupBy('Month').agg(F.sum('Revenue').alias('NetRevenue')))
by_country = retail.groupBy('Country').agg(F.sum('Revenue').alias('NetRevenue'))
top_products = (sales.filter(F.col('Revenue') > 0)
    .groupBy('StockCode').agg(F.sum('Revenue').alias('GrossRevenue'),
                              F.sum('Quantity').alias('UnitsSold'))
    .orderBy(F.desc('GrossRevenue'), F.asc('StockCode')).limit(10))

# Validate ALL outputs before any writes.
row = gold_kpis.first()
assert row is not None and gold_kpis.count() == 1
assert all(row[x] is not None for x in ['GrossRevenue','ReturnsValue','NetRevenue','TotalCustomers','TotalOrders','AOV'])
assert row['GrossRevenue'] - row['ReturnsValue'] == row['NetRevenue'], 'Revenue reconciliation mismatch'
assert monthly.filter(F.col('Month').isNull()).count() == 0
assert monthly.agg(F.sum('NetRevenue')).first()[0] == row['NetRevenue'], 'Monthly totals mismatch'
assert by_country.agg(F.sum('NetRevenue')).first()[0] == row['NetRevenue'], 'Country totals mismatch'
assert top_products.count() <= 10

outputs = {
    'gold_kpis_v2': gold_kpis,
    'gold_monthly_sales_v2': monthly,
    'gold_sales_by_country_v2': by_country,
    'gold_top_products_v2': top_products,
}
for name, frame in outputs.items():
    # Overwrite ONLY designated test Gold tables; does not append or modify Bronze/Silver.
    frame.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(f'{SCHEMA}.{name}')
    print(f'Saved {SCHEMA}.{name}: {spark.table(f"{SCHEMA}.{name}").count()} rows')
print('All Gold tables saved. KPI values:', row.asDict())



  File <command-4646846122283775>, line 13
    retail = (source.filter(F.col('InvoiceNo').isNotNull())
             ^
SyntaxError: '(' was never closed
